## Import libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from obspy import read, UTCDateTime as utc
from glob import glob

## Read waveforms and transfer S-wave picks

In [2]:
eventdir_list=sorted(glob("/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/rotated/20*"))
for eventdir in eventdir_list:
    sacfile_Z_list=sorted(glob(eventdir+"/*Z.sac"))
    for sacfile in sacfile_Z_list:
        yyyymmddhhmmss=sacfile.split("/")[-1].split(".")[0]
        network=sacfile.split("/")[-1].split(".")[-4]
        code=sacfile.split("/")[-1].split(".")[-3]
        channel=sacfile.split("/")[-1].split(".")[-2][0:2]
        
        sacfile_R=eventdir+f"/{yyyymmddhhmmss}.{network}.{code}.{channel}R.sac"
        sacfile_T=eventdir+f"/{yyyymmddhhmmss}.{network}.{code}.{channel}T.sac"
        
        tr_R, tr_T=read(sacfile_R)[0], read(sacfile_T)[0]
        
        # Check if both missing
        if not hasattr(tr_R.stats.sac, "t0") and not hasattr(tr_T.stats.sac, "t0"):
            continue

        # If R missing but T has it
        if not hasattr(tr_R.stats.sac, "t0") and hasattr(tr_T.stats.sac, "t0"):
            tr_R.stats.sac.t0=tr_T.stats.sac.t0
            tr_R.write(sacfile_R, format="SAC")

        # If T missing but R has it
        elif not hasattr(tr_T.stats.sac, "t0") and hasattr(tr_R.stats.sac, "t0"):
            tr_T.stats.sac.t0=tr_R.stats.sac.t0
            tr_T.write(sacfile_T, format="SAC")